# Cookbook
Copy-paste analysis recipes using pandas. Each section is self-contained.

**Prerequisite:** Run any earlier notebook first to create a cached parquet file.

In [6]:
from vulnflow import VulnDataset

ds = VulnDataset.from_parquet("../data/recent_cache.parquet")
df = ds.to_pandas()
print(f"{len(df):,} CVEs  |  {len(df.columns)} columns")
df.columns.tolist()

85,922 CVEs  |  23 columns


['id',
 'cve_id',
 'description',
 'nvd_published_at',
 'nvd_modified_at',
 'nvd_status',
 'cvss_version',
 'cvss_base_score',
 'cvss_base_severity',
 'cvss_source',
 'cvss_vector_string',
 'cvss_exploitability_score',
 'cvss_impact_score',
 'cvss_attack_vector',
 'cvss_attack_complexity',
 'cvss_privileges_required',
 'cvss_user_interaction',
 'cvss_scope',
 'cvss_confidentiality',
 'cvss_integrity',
 'cvss_availability',
 'tags',
 'metadata']

---
## Filter by Severity

In [7]:
critical = df[df["cvss_base_score"] >= 9.0]
high = df[df["cvss_base_severity"] == "HIGH"]
print(f"Critical (>=9.0): {len(critical):,}")
print(f"High: {len(high):,}")
critical[["cve_id", "description", "cvss_base_score"]].head(10)

Critical (>=9.0): 10,677
High: 27,773


,cve_id,description,cvss_base_score
2,CVE-2018-17207,An issue was discovered in Snap Creek Duplicat...,9.8
5,CVE-2019-19006,"Sangoma FreePBX 115.0.16.26 and below, 14.0.13...",9.8
14,CVE-2022-2421,Due to improper type validation in attachment ...,9.8
15,CVE-2022-44151,Simple Inventory Management System v1.0 is vul...,9.8
16,CVE-2022-33186,A vulnerability in Brocade Fabric OS software ...,9.8
27,CVE-2018-25095,The Duplicator WordPress plugin before 1.3.0 d...,9.8
29,CVE-2023-51951,SQL Injection vulnerability in Stock Managemen...,9.8
39,CVE-2024-2420,LenelS2 NetBox access control and event monito...,9.8
40,CVE-2024-2421,LenelS2 NetBox access control and event monito...,9.8
43,CVE-2024-37385,Roundcube Webmail before 1.5.7 and 1.6.x befor...,9.8


---
## Filter by Year

In [8]:
df["year"] = df["cve_id"].str[4:8].astype(int)
df_2024 = df[df["year"] == 2024]
print(f"{len(df_2024):,} CVEs from 2024")

39,045 CVEs from 2024


---
## Attack Vector Analysis

In [9]:
# Attack vector distribution
df["cvss_attack_vector"].value_counts()

cvss_attack_vector
N    59129
L    16040
A     1843
P      667
Name: count, dtype: int64

In [10]:
# Network-accessible, no auth required, critical severity
remotely_exploitable = df[
    (df["cvss_attack_vector"] == "NETWORK")
    & (df["cvss_privileges_required"] == "NONE")
    & (df["cvss_user_interaction"] == "NONE")
    & (df["cvss_base_score"] >= 9.0)
]
print(f"{len(remotely_exploitable):,} remotely exploitable critical CVEs")
remotely_exploitable[["cve_id", "cvss_base_score", "description"]].head(5)

0 remotely exploitable critical CVEs


,cve_id,cvss_base_score,description


---
## CVSS Impact Analysis

In [11]:
# CIA impact breakdown
for metric in ["cvss_confidentiality", "cvss_integrity", "cvss_availability"]:
    print(f"\n{metric}:")
    print(df[metric].value_counts().to_string())


cvss_confidentiality:
cvss_confidentiality
H    36968
L    23595
N    17115
P        1

cvss_integrity:
cvss_integrity
H    30971
L    25090
N    21617
P        1

cvss_availability:
cvss_availability
H    37379
N    30818
L     9481
P        1


---
## Transforms: CWE, CPE, Events
Run default transforms to extract CWE IDs, CPE vendor/product, and CVD event timestamps.

In [12]:
ds.transform(quiet=True)
df = ds.to_pandas()
print(f"{len(df.columns)} columns after transforms")

TypeError: VulnDataset.transform() got an unexpected keyword argument 'quiet'

### Top CWEs

In [ ]:
df["cwe_id"].value_counts().head(10)

### Top Vendors and Products (CPE)

In [ ]:
print("Top 10 vendors:")
print(df["cpe_vendor"].value_counts().head(10).to_string())

print("\nTop 10 products:")
print(df["cpe_product"].value_counts().head(10).to_string())

### Vendor Deep-Dive

In [ ]:
vendor = "microsoft"  # change to any vendor
vendor_df = df[df["cpe_vendor"] == vendor]
print(f"{len(vendor_df):,} CVEs for {vendor}")
print(f"\nSeverity:")
print(vendor_df["cvss_base_severity"].value_counts().to_string())
print(f"\nTop products:")
print(vendor_df["cpe_product"].value_counts().head(5).to_string())

---
## CWE Enrichment
*Requires CWEEnricher (fetches CWE database from NVD). Currently only `cwe_id` is available from NVD data; `cwe_name`, `cwe_category_id`, and `cwe_category_name` are populated after enrichment.*

In [ ]:
# TODO: CWE enrichment adds weakness names and categories
#
# from vulnflow.transforms.enrichers.cwe import CWEEnricher
# ds.transform([CWEEnricher()])
# df = ds.to_pandas()
# df[["cwe_id", "cwe_name", "cwe_category_name"]].drop_duplicates().head(10)

---
## EPSS Scores
*Requires EPSSEnricher (fetches EPSS data from FIRST.org). Adds `epss_probability`, `epss_percentile`, `epss_date`.*

In [ ]:
# TODO: EPSS enrichment adds exploitation probability scores
#
# from vulnflow.transforms.enrichers.epss import EPSSEnricher
# ds.transform([EPSSEnricher()])
# df = ds.to_pandas()
# high_epss = df[df["epss_probability"] > 0.5]
# print(f"{len(high_epss):,} CVEs with >50% exploitation probability")
# high_epss[["cve_id", "epss_probability", "epss_percentile", "cvss_base_score"]].head(10)

---
## KEV (Known Exploited Vulnerabilities)
*Requires KEVEnricher (fetches CISA KEV catalog). Adds `is_kev`, `kev_date_added`, `kev_due_date`, `kev_ransomware`.*

In [ ]:
# TODO: KEV enrichment adds known exploitation data from CISA
#
# from vulnflow.transforms.enrichers.kev import KEVEnricher
# ds.transform([KEVEnricher()])
# df = ds.to_pandas()
# kev = df[df["is_kev"] == True]
# print(f"{len(kev):,} CVEs in CISA KEV catalog")
# kev[["cve_id", "kev_date_added", "kev_due_date", "kev_ransomware"]].head(10)

---
## Metadata & Tags
*Not yet implemented. These columns are reserved for user-defined metadata and auto-generated tags.*

In [ ]:
# TODO: metadata and tags are placeholder columns
#
# metadata: dict per CVE for user-defined key-value pairs
# tags: dict per CVE for auto-generated classification tags

---
## Export to CSV

In [ ]:
df.to_csv("../data/export.csv", index=False)
print(f"Exported {len(df):,} records to data/export.csv")